# Research Question 4: Spatial Density of Reports in 2025
<div style="background-color:#f0f4f8; padding:15px; border-left:6px solid #2f5597; border-radius:6px;">

## Research Question

Where in Zurich were ZüriWieNeu reports most densely concentrated in 2025?

## Goal of this notebook

The goal of this notebook is to analyse the spatial density of ZüriWieNeu reports across Zurich in 2025 using a **Kernel Density Estimation (KDE)**.

While Research Question 3 examined the number of reports per Quartier using a choropleth map, the Quartier-level view depends strongly on the size and shape of administrative boundaries. A large Quartier may appear less intense even if reports cluster heavily in one corner of it, and small Quartiere can be visually dominated by their neighbours. A KDE removes this dependency on administrative units. It estimates a continuous density surface directly from the report locations, which makes local hotspots visible regardless of where Quartier boundaries fall.

## Why this question is relevant

A density-based perspective complements the choropleth analysis in two ways. First, it shows **where exactly** within Zurich reports cluster, rather than only which Quartier they fall into. Second, it makes it easier to see hotspots that cross Quartier borders, such as activity concentrated around a single street, square, or transport hub.

However, density values should be interpreted as a smoothed picture of reporting activity, not as a measurement of infrastructure quality. Local hotspots may reflect real infrastructure issues, but they may also be influenced by population density, nightlife, pedestrian volume, or differences in reporting behaviour.

## Planned analysis

In this notebook, I will:

1. Load and clean the ZüriWieNeu report data.
2. Filter the reports to the year 2025.
3. Convert the report coordinates into a GeoDataFrame in Swiss LV95.
4. Load the Zurich Quartiere polygon shapefile for context.
5. Run a Kernel Density Estimation on the report coordinates using `seaborn.kdeplot`.
6. Visualize the resulting density surface as a heatmap with the city outline.
7. Discuss the bandwidth choice and its effect on the result.
8. Interpret the spatial pattern of the density surface.
</div>

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from src.loading import load_csv_data
from src.cleaning import clean_reports
from src.spatial import reports_to_geodataframe

## Loading and filtering the data

The cleaning workflow defined in `src/cleaning.py` is applied to the raw ZüriWieNeu file. The reports are then filtered to the year 2025 to match the temporal scope of Research Question 3, which makes the two notebooks directly comparable.

In [ ]:
df_raw = load_csv_data("/Users/laumagoldmann/Desktop/SDS210_IndividualProject/data/raw/stzh.zwn_meldungen_p.csv")
df_clean = clean_reports(df_raw)

df_2025 = df_clean[df_clean["year"] == 2025].copy()
print(f"Reports in 2025: {len(df_2025):,}")

## Converting reports to a GeoDataFrame

The function `reports_to_geodataframe` from `src/spatial.py` builds a GeoDataFrame from the `e` and `n` coordinate columns. These coordinates are in the Swiss CH1903+ / LV95 system (EPSG:2056), which is a projected system measured in metres. This matters for the KDE: because the units are metres, the bandwidth and any distances are interpretable as real-world distances rather than as degrees of longitude and latitude.

In [ ]:
reports_2025_gdf = reports_to_geodataframe(df_2025)
reports_2025_gdf.crs

## Loading the Quartiere boundary for context

The Zurich Quartiere shapefile is loaded and reprojected to LV95 so it shares the coordinate system of the report points. For the KDE visualisation only the **outer city outline** is needed as a reference frame, which is obtained by dissolving the Quartier polygons into a single geometry.

In [ ]:
quartiere = gpd.read_file(
    "/Users/laumagoldmann/Desktop/SDS210_IndividualProject/data/raw/StatQuartiere_ZH"
).to_crs("EPSG:2056")

city_outline = quartiere.dissolve()
city_outline.head()

## Kernel Density Estimation

A Kernel Density Estimation places a small Gaussian "bump" on top of every report and then sums these bumps across the city. The result is a smooth surface that is high where many reports lie close to each other and low where reports are sparse.

The `seaborn.kdeplot` function provides a convenient 2-D KDE for point data. The most important parameter is `bw_adjust`, which scales the automatically chosen bandwidth (Scott's rule). Smaller values produce sharper, more localised hotspots; larger values produce a smoother surface. A value of `0.5` is used here as a starting point; the next section discusses the effect of this choice.

Other parameters:

- `fill=True` and `cmap="rocket_r"` produce a filled, dark-to-light heatmap.
- `levels=20` controls how many density bands are drawn.
- `thresh=0.05` makes the lowest 5 % of density transparent so the city outline shows through in low-activity areas.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

sns.kdeplot(
    data=reports_2025_gdf,
    x=reports_2025_gdf.geometry.x,
    y=reports_2025_gdf.geometry.y,
    fill=True,
    cmap="rocket_r",
    levels=20,
    thresh=0.05,
    bw_adjust=0.5,
    alpha=0.85,
    ax=ax,
)

# City outline for spatial reference
city_outline.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1.0
)

# Individual report points, very faint
ax.scatter(
    reports_2025_gdf.geometry.x,
    reports_2025_gdf.geometry.y,
    s=1,
    color="black",
    alpha=0.1
)

ax.set_title(
    "Kernel Density of ZüriWieNeu Reports in Zurich, 2025",
    fontsize=14
)
ax.set_xlabel("Easting (LV95, m)")
ax.set_ylabel("Northing (LV95, m)")
ax.set_aspect("equal")

minx, miny, maxx, maxy = city_outline.total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

plt.tight_layout()
plt.savefig(
    "/Users/laumagoldmann/Desktop/SDS210_IndividualProject/outputs/R4_KDE_Heatmap_2025.png",
    bbox_inches="tight",
    dpi=200
)
plt.show()

## Sensitivity to the bandwidth

The bandwidth is the single most important choice in a KDE. To make this choice visible, the same KDE is computed for three values of `bw_adjust`: a sharp setting (`0.3`), the value used above (`0.5`), and a smoother setting (`1.0`). The smaller value emphasises individual hotspots, while the larger one shows the overall pattern across the city.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7), sharex=True, sharey=True)

bw_values = [0.3, 0.5, 1.0]
labels = ["Sharper (bw_adjust = 0.3)", "Used above (bw_adjust = 0.5)", "Smoother (bw_adjust = 1.0)"]

for ax, bw, label in zip(axes, bw_values, labels):
    sns.kdeplot(
        data=reports_2025_gdf,
        x=reports_2025_gdf.geometry.x,
        y=reports_2025_gdf.geometry.y,
        fill=True,
        cmap="rocket_r",
        levels=20,
        thresh=0.05,
        bw_adjust=bw,
        alpha=0.85,
        ax=ax,
    )
    city_outline.boundary.plot(ax=ax, color="black", linewidth=0.8)
    ax.set_title(label, fontsize=12)
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)
    ax.set_aspect("equal")
    ax.axis("off")

plt.suptitle("Effect of bandwidth on the KDE surface", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

<div style="background-color:#eef6ff; padding:15px; border-left:6px solid #4a90e2; border-radius:6px;">

## Interpretation of the density surface

The heatmap shows the estimated spatial density of ZüriWieNeu reports across Zurich in 2025. Unlike a Quartier choropleth, the surface is continuous: it shows where reports cluster regardless of administrative boundaries.

The strongest concentration of reports appears in the **central area of Zurich**, in particular around **Langstrasse**, **Sihlfeld**, **Werd**, and **Hard**. This matches the pattern found in Research Question 3, where the same Quartiere had the highest absolute report counts. The KDE adds an important detail: within these Quartiere, density is not evenly distributed but concentrated along specific corridors, suggesting that reporting activity follows particular streets and squares rather than entire neighbourhoods.

Secondary hotspots can be seen in **Wipkingen**, **Unterstrass**, and parts of **Altstetten**. Outer Quartiere such as **Witikon**, **Schwamendingen-Mitte**, or **Affoltern** show very low densities, indicating sparser reporting activity in these areas.

The bandwidth comparison illustrates how strongly the visual impression depends on the smoothing choice. A small bandwidth makes individual streets and squares visible but can give the impression of more, smaller hotspots than the data really supports. A large bandwidth shows a single broad central cluster but hides any internal structure. The intermediate value used in the main map balances the two perspectives.

As with the choropleth analysis, the density values should be interpreted as a representation of **reporting activity**, not as a direct measure of infrastructure quality. Hotspots may reflect actual problem concentrations, but they are also likely shaped by pedestrian volume, nightlife, commercial activity, and population density.
</div>

<div style="background-color:#eef9f0; padding:15px; border-left:6px solid #3c9d5d; border-radius:6px;">

## Conclusion

This notebook analysed the spatial density of ZüriWieNeu reports across Zurich in 2025 using a Kernel Density Estimation.

The KDE produced a continuous density surface that is independent of Quartier boundaries. The strongest hotspot lies in the central part of the city, around **Langstrasse**, **Sihlfeld**, **Werd**, and **Hard**, with secondary clusters in **Wipkingen**, **Unterstrass**, and parts of **Altstetten**. Outer Quartiere show clearly lower densities.

The bandwidth sensitivity check showed that the level of smoothing strongly affects the visual result. A `bw_adjust` value of `0.5` was chosen as a compromise between resolving local hotspots and showing the overall city-wide pattern.

The KDE results complement the Quartier-level choropleth from Research Question 3. The choropleth gives a clear administrative summary, while the KDE shows where within those Quartiere the reporting activity is actually concentrated. Taken together, the two analyses confirm that ZüriWieNeu reporting in 2025 was strongly concentrated in central Zurich, both at the Quartier level and at the finer scale of streets and squares.

As with the previous spatial analyses, the findings describe **patterns of reporting activity** rather than direct measurements of urban infrastructure quality.
</div>